In [ ]:
# Cell 1 — Environment, paths, and dependencies

import shutil
import subprocess
import sys
from pathlib import Path

# ------------------------------------------------------------
# Detect Google Colab
# ------------------------------------------------------------

try:
    import google.colab  # type: ignore
    IS_COLAB = True
except ImportError:
    IS_COLAB = False

print(f"Environment: {'Google Colab' if IS_COLAB else 'Local Jupyter / Python'}")

# ------------------------------------------------------------
# Define the security_frameworks directory
# ------------------------------------------------------------

if IS_COLAB:
    # Colab's working area
    SECURITY_FRAMEWORKS_DIR = Path("/content/security_frameworks")
else:
    # When running locally, the notebook is expected to be
    # launched from the security_frameworks directory. Guard
    # against an accidental launch from elsewhere: if the cwd
    # doesn't contain this notebook, every relative assumption
    # downstream (REPO_DIR, OUTPUT_DIR) would silently point at
    # the wrong place.
    SECURITY_FRAMEWORKS_DIR = Path.cwd()

    _this_notebook = SECURITY_FRAMEWORKS_DIR / "MITRE_ATTACK_git_exploration.ipynb"
    if not _this_notebook.exists():
        raise RuntimeError(
            "This notebook must be launched from the security_frameworks "
            f"directory. Current working directory is {SECURITY_FRAMEWORKS_DIR}, "
            f"which does not contain {_this_notebook.name}."
        )

# Create the directory if necessary
SECURITY_FRAMEWORKS_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# Repository and output locations
# ------------------------------------------------------------

REPO_DIR = SECURITY_FRAMEWORKS_DIR / "MITRE_ATTACK"
OUTPUT_DIR = SECURITY_FRAMEWORKS_DIR / "MITRE_ATTACK_output"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

REPO_URL = "https://github.com/mitre-attack/attack-stix-data.git"

# Pin retrieval to a specific ATT&CK release, e.g. "17.1".
# When None, the unversioned "latest" file per domain is used
# (the repo's own moving pointer to the current release). Set
# this to reproduce a past run or to pin a specific version for
# a briefing; the pinned version is validated to exist per
# domain in the next cells, and is recorded in manifest.json.
PIN = None

print(f"Security frameworks directory: {SECURITY_FRAMEWORKS_DIR}")
print(f"Repository:                    {REPO_DIR}")
print(f"Output directory:              {OUTPUT_DIR}")
print(f"Pinned version:                {PIN if PIN else '(none — using latest)'}")

# ------------------------------------------------------------
# Check Git
# ------------------------------------------------------------

GIT = shutil.which("git")

if GIT is None:
    raise RuntimeError(
        "Git is not installed or is not available on PATH."
    )

print(f"\n✓ Git: {GIT}")

In [ ]:
# Cell 2 — Clone or update MITRE ATTACK STIX Data

if not (REPO_DIR / ".git").exists():

    print("Downloading MITRE ATTACK STIX Data...")

    subprocess.run(
        [
            GIT,
            "clone",
            "--depth",
            "1",
            REPO_URL,
            str(REPO_DIR),
        ],
        check=True,
    )

else:

    print("MITRE ATTACK repository already exists. Updating...")

    subprocess.run(
        [
            GIT,
            "-C",
            str(REPO_DIR),
            "fetch",
            "origin",
            "--depth",
            "1",
        ],
        check=True,
    )

    # Determine the repository's default branch.
    result = subprocess.run(
        [
            GIT,
            "-C",
            str(REPO_DIR),
            "symbolic-ref",
            "--quiet",
            "--short",
            "refs/remotes/origin/HEAD",
        ],
        capture_output=True,
        text=True,
        check=False,
    )

    remote_head = result.stdout.strip()

    if not remote_head:
        remote_head = "origin/master" # ATTACK default branch fallback

    subprocess.run(
        [
            GIT,
            "-C",
            str(REPO_DIR),
            "reset",
            "--hard",
            remote_head,
        ],
        check=True,
    )

# ------------------------------------------------------------
# Record exact Git commit
# ------------------------------------------------------------

commit_result = subprocess.run(
    [
        GIT,
        "-C",
        str(REPO_DIR),
        "rev-parse",
        "HEAD",
    ],
    capture_output=True,
    text=True,
    check=True,
)

git_commit = commit_result.stdout.strip()

print("\n" + "=" * 70)
print("MITRE ATTACK repository ready")
print("=" * 70)
print(f"Repository : {REPO_DIR}")
print(f"Git commit : {git_commit}")
print(f"Output     : {OUTPUT_DIR}")

In [ ]:
# Cell 3 — Locate, version-pin, and copy ATT&CK STIX bundles to Output Directory

import hashlib
import json
import shutil
from datetime import datetime, timezone

DOMAINS = {
    "enterprise": "enterprise-attack",
    "ics": "ics-attack",
    "mobile": "mobile-attack"
}

# ------------------------------------------------------------
# Confirm whether the clone ships versioned per-release files
# (e.g. enterprise-attack/enterprise-attack-17.1.json), in
# addition to the unversioned "latest" file
# (enterprise-attack/enterprise-attack.json). PIN can only be
# honoured if these exist.
# ------------------------------------------------------------

_probe_folder = next(iter(DOMAINS.values()))
_probe_dir = REPO_DIR / _probe_folder
_versioned_candidates = sorted(_probe_dir.glob(f"{_probe_folder}-*.json"))
HAS_VERSIONED_FILES = len(_versioned_candidates) > 0
_index_path = REPO_DIR / "index.json"

print(f"Versioned per-release files present ({_probe_folder}): {HAS_VERSIONED_FILES}")
if HAS_VERSIONED_FILES:
    print(f"  e.g. {_versioned_candidates[0].name} .. {_versioned_candidates[-1].name} "
          f"({len(_versioned_candidates)} files)")
print(f"Repo-level index.json present: {_index_path.exists()}")

if PIN is not None and not HAS_VERSIONED_FILES:
    raise RuntimeError(
        f"PIN={PIN!r} was requested, but the cloned repository does not ship "
        "versioned per-release files (e.g. enterprise-attack-<ver>.json). "
        "Cannot honour the pin."
    )

# ------------------------------------------------------------
# Locate the source bundle per domain (pinned or latest),
# load it, and read its x-mitre-collection version.
# ------------------------------------------------------------

JSON_FILES = {}   # domain -> flat copy path in OUTPUT_DIR
BUNDLES = {}       # domain -> parsed STIX bundle (for manifest counts)
VERSIONS = {}      # domain -> x_mitre_version string

for domain, folder in DOMAINS.items():
    domain_dir = REPO_DIR / folder

    if PIN is not None:
        src_path = domain_dir / f"{folder}-{PIN}.json"
        if not src_path.exists():
            raise FileNotFoundError(
                f"PIN={PIN!r} was requested but the versioned bundle was not found:\n{src_path}"
            )
    else:
        src_path = domain_dir / f"{folder}.json"
        if not src_path.exists():
            raise FileNotFoundError(
                f"Required ATT&CK bundle was not found:\n{src_path}"
            )

    with open(src_path, "r", encoding="utf-8") as f:
        bundle = json.load(f)

    collections = [
        obj for obj in bundle.get("objects", [])
        if obj.get("type") == "x-mitre-collection"
    ]
    if not collections:
        raise ValueError(f"No x-mitre-collection object found in {src_path}")

    version = collections[0].get("x_mitre_version")
    if not version:
        raise ValueError(
            f"x-mitre-collection in {src_path} has no x_mitre_version"
        )

    BUNDLES[domain] = bundle
    VERSIONS[domain] = version

    # Flat copy in the centralized output directory (existing behaviour).
    dest_path = OUTPUT_DIR / f"{folder}.json"
    shutil.copy2(src_path, dest_path)
    JSON_FILES[domain] = dest_path
    print(f"✓ Copied {domain} bundle (x_mitre_version={version}) to: {dest_path}")

# ------------------------------------------------------------
# Also write versioned copies under MITRE_ATTACK_output/<version>/.
#
# ATT&CK's three domains are versioned independently — enterprise,
# ics and mobile releases do not always share the same
# x_mitre_version. A single shared "<version>/" folder is only
# unambiguous when all three domains report the same version; if
# they differ, per-domain subfolders ("<version>/<folder>/") are
# used instead, so one domain's bundle can never silently land in,
# or overwrite, another domain's version folder.
# ------------------------------------------------------------

distinct_versions = set(VERSIONS.values())
SHARED_VERSION_LAYOUT = len(distinct_versions) == 1

VERSIONED_PATHS = {}

for domain, folder in DOMAINS.items():
    version = VERSIONS[domain]
    if SHARED_VERSION_LAYOUT:
        version_dir = OUTPUT_DIR / version
    else:
        version_dir = OUTPUT_DIR / version / folder
    version_dir.mkdir(parents=True, exist_ok=True)

    versioned_dest = version_dir / f"{folder}.json"
    shutil.copy2(JSON_FILES[domain], versioned_dest)
    VERSIONED_PATHS[domain] = versioned_dest
    print(f"✓ Versioned copy: {versioned_dest}")

print("\n" + "=" * 70)
print("ATT&CK STIX BUNDLES LOCATED & COPIED TO OUTPUT")
print(f"Version layout: {'shared' if SHARED_VERSION_LAYOUT else 'per-domain'} "
      f"({', '.join(f'{d}={v}' for d, v in VERSIONS.items())})")
print("=" * 70)

# ------------------------------------------------------------
# Build manifest.json: provenance + per-domain object counts.
# ------------------------------------------------------------

def sha256_of(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

def count_objects(bundle):
    counts = {
        "technique": 0,
        "sub_technique": 0,
        "x-mitre-tactic": 0,
        "course-of-action": 0,
        "intrusion-set": 0,
        "malware_and_tool": 0,
        "campaign": 0,
        "x-mitre-data-source": 0,
        "revoked": 0,
        "deprecated": 0,
    }
    for obj in bundle.get("objects", []):
        obj_type = obj.get("type")
        if obj_type == "attack-pattern":
            if obj.get("x_mitre_is_subtechnique"):
                counts["sub_technique"] += 1
            else:
                counts["technique"] += 1
        elif obj_type == "x-mitre-tactic":
            counts["x-mitre-tactic"] += 1
        elif obj_type == "course-of-action":
            counts["course-of-action"] += 1
        elif obj_type == "intrusion-set":
            counts["intrusion-set"] += 1
        elif obj_type in ("malware", "tool"):
            counts["malware_and_tool"] += 1
        elif obj_type == "campaign":
            counts["campaign"] += 1
        elif obj_type == "x-mitre-data-source":
            counts["x-mitre-data-source"] += 1

        if obj.get("revoked") is True:
            counts["revoked"] += 1
        if obj.get("x_mitre_deprecated") is True:
            counts["deprecated"] += 1

    return counts

manifest = {
    "repo_url": REPO_URL,
    "git_commit": git_commit,
    "pin": PIN,
    "retrieved_at_utc": datetime.now(timezone.utc).isoformat(),
    "domains": {},
}

for domain, folder in DOMAINS.items():
    counts = count_objects(BUNDLES[domain])

    if counts["technique"] == 0:
        raise ValueError(
            f"{domain} bundle ({JSON_FILES[domain]}) reports zero techniques — "
            "refusing to treat this as a valid ATT&CK release."
        )

    manifest["domains"][domain] = {
        "file": str(JSON_FILES[domain].relative_to(OUTPUT_DIR)),
        "versioned_file": str(VERSIONED_PATHS[domain].relative_to(OUTPUT_DIR)),
        "x_mitre_version": VERSIONS[domain],
        "sha256": sha256_of(JSON_FILES[domain]),
        "counts": counts,
    }

manifest_path = OUTPUT_DIR / "manifest.json"
with open(manifest_path, "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2, ensure_ascii=False)

print(f"\n✓ Wrote manifest: {manifest_path}")
print(json.dumps(manifest, indent=2, ensure_ascii=False))

In [ ]:
# Cell 4 — Extract and print the inferred JSON structure

import json
from collections import OrderedDict

# ------------------------------------------------------------
# Schema inference utilities
# ------------------------------------------------------------

def json_type(value):
    """Return the JSON data type without exposing the actual value."""
    if value is None: return "null"
    if isinstance(value, bool): return "boolean"
    if isinstance(value, int) and not isinstance(value, bool): return "integer"
    if isinstance(value, float): return "number"
    if isinstance(value, str): return "string"
    if isinstance(value, list): return "array"
    if isinstance(value, dict): return "object"
    return "unknown"

def merge_types(existing, new_type):
    """Merge multiple observed JSON types."""
    if existing is None: return new_type
    existing_types = existing if isinstance(existing, list) else [existing]
    new_types = new_type if isinstance(new_type, list) else [new_type]
    merged = []
    for t in existing_types + new_types:
        if t not in merged: merged.append(t)
    return merged[0] if len(merged) == 1 else sorted(merged)

def schema_from_value(value):
    """
    Build a value-free schema:
    - objects -> fields are recursively represented
    - arrays -> only the item schema/type is represented
    - primitives -> type only
    """
    value_type = json_type(value)

    if value_type == "object":
        fields = OrderedDict()
        for key in sorted(value.keys()):
            fields[key] = schema_from_value(value[key])
        return {"type": "object", "fields": fields}

    if value_type == "array":
        if not value: return {"type": "array", "items": {}}
        item_schema = None
        for item in value:
            current = schema_from_value(item)
            if item_schema is None:
                item_schema = current
            elif item_schema.get("type") == "object" and current.get("type") == "object":
                item_schema = merge_object_schemas(item_schema, current)
            else:
                item_schema = merge_schema_nodes(item_schema, current)
        return {"type": "array", "items": item_schema}

    return {"type": value_type}

def merge_schema_nodes(a, b):
    """Merge two schema nodes while retaining only field/type information."""
    a_type = a.get("type")
    b_type = b.get("type")

    if a_type == "object" and b_type == "object":
        return merge_object_schemas(a, b)

    if a_type == "array" and b_type == "array":
        if not a.get("items"): return b
        if not b.get("items"): return a
        return {"type": "array", "items": merge_schema_nodes(a["items"], b["items"])}

    return {"type": merge_types(a_type, b_type)}

def merge_object_schemas(a, b):
    """Merge fields from multiple object instances/shapes."""
    merged_fields = OrderedDict()
    for key in sorted(set(a.get("fields", {})) | set(b.get("fields", {}))):
        if key in a.get("fields", {}) and key in b.get("fields", {}):
            merged_fields[key] = merge_schema_nodes(a["fields"][key], b["fields"][key])
        elif key in a.get("fields", {}):
            merged_fields[key] = a["fields"][key]
        else:
            merged_fields[key] = b["fields"][key]
    return {"type": "object", "fields": merged_fields}

def extract_schema(path):
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)
    return schema_from_value(data)

# ------------------------------------------------------------
# Analyze ATTACK STIX files
# ------------------------------------------------------------

schemas = OrderedDict()

for domain, json_file in JSON_FILES.items():
    print(f"Analyzing: {json_file}")
    schemas[domain] = extract_schema(json_file)

# ------------------------------------------------------------
# Print inferred structures
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("INFERRED JSON STRUCTURE")
print("=" * 70)

print(
    json.dumps(
        schemas,
        indent=2,
        ensure_ascii=False,
    )
)